# 완성 정면 캐릭터 → 공식 See-through → PSD → Cubism PRO 모델 템플릿 인계

**이 Colab에 FACE 단독 원화를 넣으면 안 됩니다.**

권장 입력은 **`complete_character__ARTWORK.zip` 한 개**입니다. ZIP 안에는 옷과 헤어를 갖춘 **완성된 여성 캐릭터 정면 그림** `outputs/master.png`가 있어야 합니다. 또는 동일한 완성 그림 `full_character_input.png` 한 장만 업로드합니다.

정확한 등록 캔버스: **1536×2304 RGBA**. 기존 `face__ARTWORK.zip`의 1536×1920 FACE가 화면 높이의 94%를 차지해 의상과 자동 정렬되지 않는 것이 주요 문제였습니다. FACE 원본을 재사용하려면 `face_registered_upper.png`(0.55배, 등록 범위 X380–1156/Y140–1132)를 기준으로 OUTFIT과 HAIR를 **이미지 편집 방식**으로 만들어야 합니다.

고급 합성 입력: `face_registered_upper.png`, `outfit.png`, `hair.png`, `registration_contract.json` 4개를 한꺼번에 업로드. 크기가 같아도 실제 머리·목 연결은 사람이 확인해야 합니다. 마스크는 첫 분리에서 빼고 이후 별도 ACCESSORY로 처리합니다.

원화 참고/생성용 ZIP은 See-through 추론에 그대로 업로드하지 않습니다. GPU 및 공식 CLI 실행 성공은 별도 확인이 필요합니다.


## 1. 공식 런타임 환경 검사 + 설치

공식 문서의 Python **3.12**, PyTorch **2.8.0+cu128**, torchvision **0.23.0+cu128**, torchaudio **2.8.0+cu128**, `requirements.txt`를 기준으로 합니다. 다른 버전이 반드시 실패한다는 뜻은 아니지만, **공식 동일 설정**으로 판정하지 않습니다.

Python 버전이 다르면 런타임 자체를 Python 3.12로 다시 구성해야 합니다. pip만으로 커널 Python 버전을 바꿀 수 없습니다. PyTorch가 이미 커널에 import되어 설치 버전을 교체했다면 런타임을 재시작하고 처음부터 실행하십시오.

In [ ]:
import sys, subprocess, os, json, shutil, platform, importlib.metadata as md
from pathlib import Path
ROOT=Path('/content')
REPO=ROOT/'see-through'
OUTPUT=ROOT/'face_see_through_v3_output'
OUTPUT.mkdir(parents=True,exist_ok=True)
if not (sys.version_info.major==3 and sys.version_info.minor==12):
    raise RuntimeError(f'OFFICIAL PYTHON MISMATCH: {sys.version.split()[0]} — 공식은 Python 3.12. 3.12 런타임으로 변경하세요.')
if not shutil.which('nvidia-smi'):
    raise RuntimeError('NVIDIA CUDA GPU runtime required (nvidia-smi missing). Colab GPU를 선택하세요.')
subprocess.run(['nvidia-smi'],check=True)
def installed(pkg):
    try:return md.version(pkg)
    except md.PackageNotFoundError:return None
wanted={'torch':'2.8.0+cu128','torchvision':'0.23.0+cu128','torchaudio':'2.8.0+cu128'}
actual={k:installed(k) for k in wanted}
print('Current PyTorch packages:',actual)
if any(actual[k]!=v for k,v in wanted.items()):
    if any(x in sys.modules for x in ('torch','torchvision','torchaudio')):
        raise RuntimeError('PyTorch imported already. Restart runtime before installing pinned official versions.')
    cmd=[sys.executable,'-m','pip','install','--upgrade',*(f'{k}=={v}' for k,v in wanted.items()),'--index-url','https://download.pytorch.org/whl/cu128']
    subprocess.run(cmd,check=True)
    print('Torch packages installed. If Colab already loaded torch internally, restart runtime before inference.')
print('Official Python/PyTorch wheel preflight complete; CUDA runtime health checked after package install.')

## 2. 공식 저장소 설치 + 정확한 버전 기록

`main` 최신 커밋은 바뀔 수 있으므로 SHA를 기록합니다. 설치 전에 사용자가 확인해야 하는 저장소이며, `assets → common/assets` 링크를 공식 지침대로 구성합니다.

In [ ]:
from pathlib import Path
import subprocess,sys,os,json
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/shitagaki-lab/see-through.git',str(REPO)],check=True)
sha=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('OFFICIAL_REPO_SHA =',sha)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],cwd=REPO,check=True)
assetlink=REPO/'assets'
if not assetlink.exists() and not assetlink.is_symlink():
    assetlink.symlink_to(Path('common/assets'))
if not assetlink.exists():
    raise RuntimeError('Official assets symlink missing or target common/assets unavailable')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available after official package installation. Check driver/runtime and restart.')
VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
print('PyTorch',torch.__version__,'CUDA',torch.version.cuda,'GPU',torch.cuda.get_device_name(0),'VRAM_GiB',round(VRAM,2))
if not (torch.__version__.startswith('2.8.0+cu128')):
    raise RuntimeError(f'PyTorch changed during install: {torch.__version__}; expected official 2.8.0+cu128. Restart/repair before continuing')
if VRAM < 8:
    raise RuntimeError('8 GiB 미만 GPU는 이 노트북의 공식 문서 기반 권장 프로파일 밖입니다.')

## 3. 정확히 어떤 파일을 업로드하나요?

**권장 — 하나만 업로드:** `complete_character__ARTWORK.zip` 안의 `outputs/master.png` (머리카락 + 민트눈 FACE + 옷 입은 상반신/팔까지 포함된 투명 PNG, 정확히 1536×2304).

대안 — `full_character_input.png` (같은 조건의 완성 정면 한 장).

**정말 자산을 따로 생성할 경우:** 이미 등록된 `face_registered_upper.png`와 동일 캔버스에 실제 그린 `outfit.png`·`hair.png`·`registration_contract.json`을 한꺼번에 올립니다. FACE 원래 ZIP (1536×1920)을 직접 올리면 실패합니다.

`reference_face_on_light.jpg`, `guide_outfit.jpg`, `guide_hair.jpg` 등의 파일은 **이미지 생성 시의 위치 참고용**이지 See-through 입력이 아닙니다. 의상 없이 얼굴만 업로드하면 중단됩니다.


In [ ]:
import io, zipfile, hashlib, json, numpy as np
from PIL import Image
from google.colab import files
from IPython.display import display
uploads=files.upload()
if not uploads: raise ValueError('입력이 없습니다. complete_character__ARTWORK.zip 하나를 권장합니다.')
SIZE=(1536,2304)  # registered UPPER character, NOT original 1536x1920 FACE

def strict_rgba(raw, label):
    im=Image.open(io.BytesIO(raw)); im.load()
    if im.format!='PNG' or im.mode!='RGBA' or im.size!=SIZE:
        raise ValueError(f'{label}: expected original PNG RGBA {SIZE}; got {im.format}/{im.mode}/{im.size}; do not auto-resize')
    amin,amax=im.getchannel('A').getextrema()
    if (amin,amax)!=(0,255):
        raise ValueError(f'{label}: requires genuine transparent background and opaque artwork')
    return im.copy()

def check_complete(im, label):
    box=im.getchannel('A').getbbox()
    if box is None or box[1]>int(.16*SIZE[1]) or box[3]<int(.84*SIZE[1]):
        raise ValueError(f'{label}: appears FACE-only or incomplete upper character: alpha bbox {box}')
    return box

if 'complete_character__ARTWORK.zip' in uploads:
    if len(uploads)!=1: raise ValueError('Upload only one complete character ZIP')
    with zipfile.ZipFile(io.BytesIO(uploads['complete_character__ARTWORK.zip'])) as archive:
        if archive.testzip() is not None: raise ValueError('ZIP integrity failure')
        names=archive.namelist()
        if names.count('outputs/master.png')!=1: raise ValueError('Exactly one outputs/master.png required')
        info=archive.getinfo('outputs/master.png')
        if not 0<info.file_size<=48_000_000: raise ValueError('Invalid master PNG size')
        full=strict_rgba(archive.read(info),'complete_character__ARTWORK.zip/outputs/master.png')
        if 'metadata/registration_contract.json' in names:
            meta=json.loads(archive.read('metadata/registration_contract.json').decode())
            if meta.get('canvas')!=list(SIZE): raise ValueError('Registration contract canvas mismatch')
    INPUT_MODE='COMPLETE_CHARACTER_ZIP'
elif 'full_character_input.png' in uploads:
    if len(uploads)!=1: raise ValueError('Upload only the complete character PNG')
    full=strict_rgba(uploads['full_character_input.png'],'full_character_input.png')
    INPUT_MODE='COMPLETE_CHARACTER_PNG'
else:
    required={'face_registered_upper.png','outfit.png','hair.png','registration_contract.json'}
    missing=required-set(uploads)
    if missing: raise ValueError('FACE alone cannot be separated. Missing: '+', '.join(sorted(missing)))
    contract=json.loads(uploads['registration_contract.json'].decode('utf-8'))
    if contract.get('canvas')!=list(SIZE): raise ValueError('Registered canvas mismatch')
    if hashlib.sha256(uploads['face_registered_upper.png']).hexdigest()!=contract.get('registered_face_sha256'):
        raise ValueError('Registered FACE was modified. Preserve supplied face_registered_upper.png exactly.')
    face=strict_rgba(uploads['face_registered_upper.png'],'REGISTERED FACE')
    outfit=strict_rgba(uploads['outfit.png'],'OUTFIT overlay')
    hair=strict_rgba(uploads['hair.png'],'HAIR overlay')
    ob=outfit.getchannel('A').getbbox();hb=hair.getchannel('A').getbbox()
    if ob is None or ob[3]<int(.84*SIZE[1]) or ob[1]>int(.57*SIZE[1]):
        raise ValueError(f'OUTFIT not located on upper torso: {ob}')
    if hb is None or hb[1]>int(.25*SIZE[1]) or hb[0]>SIZE[0]/2 or hb[2]<SIZE[0]/2:
        raise ValueError(f'HAIR not located on head: {hb}')
    full=Image.new('RGBA',SIZE,(0,0,0,0))
    for layer in (outfit,face,hair): full.alpha_composite(layer)
    INPUT_MODE='REGISTERED_FACE_OUTFIT_HAIR'
bbox=check_complete(full,INPUT_MODE)
input_path=ROOT/'full_character_input.png'
full.save(input_path)
print('INPUT_MODE',INPUT_MODE,'canvas',full.size,'alpha bbox',bbox)
display(full.resize((480,int(480*full.height/full.width))))
print('Inspect visual anatomy: same female/eyes, haircut fit, seamless neck/collar, shoulders and arms. Filename/alpha checks are NOT proof of alignment.')


## 4. 정면 캐릭터 육안 검수

눈이 큰 민트색 여성 FACE, 검정/흰색 헤어, 옷을 입은 몸체와 어깨/팔까지 **한 그림**에 보여야 합니다. 옷깃과 목·머리카락이 맞는지 직접 확인하십시오. 얼굴만 나왔다면 `VISUAL_APPROVED=False`를 유지하고 원화를 다시 만드십시오. 모델 PSD의 리깅 성공 판정은 별개입니다.


In [ ]:
VISUAL_APPROVED=False  # 위 프리뷰를 보고 완성된 캐릭터임을 확인한 뒤 True 로 변경
if not VISUAL_APPROVED:
    raise RuntimeError('Full character/alignment preview not approved. Review preview before inference.')

## 5. 공식 CLI 플래그와 VRAM 프로파일 검사

공식 인자 존재 여부를 `ast`로 파싱해 **모델 로딩 전에** 확인합니다. 스크립트가 업데이트되어 인터페이스가 바뀌었으면 진행을 중단합니다. `--tblr_split` 좌우 눈/귀 등의 분리를 시도하나 성공 보장은 아닙니다.

In [ ]:
import ast
from pathlib import Path
assert VISUAL_APPROVED
if VRAM >= 16:
    MODE='bf16'; script='inference/scripts/inference_psd.py'; resolution=1280; offload=False
elif VRAM >= 10:
    MODE='bf16_group_offload'; script='inference/scripts/inference_psd.py'; resolution=1280; offload=True
else:
    MODE='nf4'; script='inference/scripts/inference_psd_quantized.py'; resolution=1024 if VRAM<9 else 1280; offload=False
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements-inference-bnb.txt'],cwd=REPO,check=True)
script_file=REPO/script
if not script_file.exists():raise FileNotFoundError(f'Official script missing: {script_file}')
source=script_file.read_text(encoding='utf-8')
tree=ast.parse(source)
flags=set()
for node in ast.walk(tree):
    if isinstance(node,ast.Call) and isinstance(node.func,ast.Attribute) and node.func.attr=='add_argument':
        for v in node.args:
            if isinstance(v,ast.Constant) and isinstance(v.value,str) and v.value.startswith('--'):flags.add(v.value)
required={'--srcp','--save_to_psd','--save_dir','--resolution','--tblr_split'}
if offload:required.add('--group_offload')
missing=required-flags
if missing:raise RuntimeError(f'OFFICIAL CLI CHANGED; missing {missing} in {script_file}. Halt before GPU inference.')
SAVE_DIR=OUTPUT/'official_output'
SAVE_DIR.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,script,'--srcp',str(input_path),'--save_to_psd','--save_dir',str(SAVE_DIR),'--tblr_split','--resolution',str(resolution),'--seed','42']
if offload:cmd.append('--group_offload')
run_meta={'official_git_sha':sha,'mode':MODE,'vram_gib':round(VRAM,2),'resolution':resolution,'command':cmd,
          'input_mode':INPUT_MODE,'full_character_preview_human_approved':VISUAL_APPROVED,
          'cubism_verified':False,'see_through_psd_verified':False}
(OUTPUT/'run_meta.json').write_text(json.dumps(run_meta,ensure_ascii=False,indent=2))
print('Official profile',MODE,'script',script,'resolution',resolution,'flags passed')

## 6. GPU 추론: 원본 공식 파이프라인만 실행

기본 스크립트가 **두 단계 LayerDiff + Marigold + PSD**를 실행합니다. 추론 중 오류는 숨기지 않고 로그를 보존합니다. 처음 가중치 다운로드 시 시간이 더 소요될 수 있습니다.

In [ ]:
import time, subprocess
start=time.monotonic()
print('RUNNING:', ' '.join(cmd))
with (OUTPUT/'see_through_stdout.log').open('w',encoding='utf-8') as out, (OUTPUT/'see_through_stderr.log').open('w',encoding='utf-8') as err:
    proc=subprocess.run(cmd,cwd=REPO,stdout=out,stderr=err,text=True)
run_meta['exit_code']=proc.returncode
run_meta['elapsed_seconds']=round(time.monotonic()-start,2)
(OUTPUT/'run_meta.json').write_text(json.dumps(run_meta,ensure_ascii=False,indent=2))
if proc.returncode:
    print((OUTPUT/'see_through_stderr.log').read_text(encoding='utf-8')[-5000:])
    raise RuntimeError(f'See-through failed with exit {proc.returncode}; diagnostics in {OUTPUT}')
psds=list(SAVE_DIR.rglob('*.psd'))
if not psds:raise RuntimeError('Exit code 0 but no official PSD produced; not a successful run')
print('Found official PSDs:',[str(p) for p in psds])

## 7. PSD 구조/이미지 실체 검사 + 로그·ZIP 다운로드

이 검사는 **PSD가 실제 여러 레이어로 열리는지**만 확인합니다. 눈 뒤 흰자와 입안 인페인팅 품질·Live2D 메시·움직임은 별도 육안 및 Cubism 검증이 필수입니다.

In [ ]:
from psd_tools import PSDImage
import hashlib,zipfile,json,shutil
qa={'repo_sha':sha,'mode':MODE,'psds':[],'cubism_test':'NOT_RUN','rigging_usable':'UNVERIFIED'}
def digest(path):
    d=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(2**20),b''):d.update(block)
    return d.hexdigest()
for p in psds:
    psd=PSDImage.open(p)
    nodes=[]
    def inspect(items,depth=0):
        for a in items:
            nodes.append({'name':str(a.name),'depth':depth,'visible':bool(a.is_visible()),'group':bool(a.is_group())})
            if a.is_group():inspect(a,depth+1)
    inspect(psd)
    non_groups=[v for v in nodes if not v['group']]
    if len(non_groups)<2:raise RuntimeError(f'PSD appears flattened: {p}')
    if psd.width<128 or psd.height<128:raise RuntimeError(f'Invalid PSD dimensions: {p}')
    qa['psds'].append({'file':str(p.relative_to(SAVE_DIR)),'bytes':p.stat().st_size,'size':list(psd.size),
       'leaf_layers':len(non_groups),'layers':nodes,'sha256':digest(p)})
qa['structure_pass']=bool(qa['psds'])
qa['remaining_human_checks']=['Eye whites exposed when pupils move','Top/bottom lashes and lid coverage during blinks',
                              'Mouth interior during talking','Deformer and ArtMesh in Cubism','45° turn mesh does not reveal gaps']
(OUTPUT/'PSD_STRUCTURE_QA.json').write_text(json.dumps(qa,ensure_ascii=False,indent=2),encoding='utf-8')
shutil.copy2(input_path,OUTPUT/'full_character_input.png')
run_meta['see_through_psd_verified']=True
(OUTPUT/'run_meta.json').write_text(json.dumps(run_meta,ensure_ascii=False,indent=2))
archive=ROOT/'SeeThrough_OFFICIAL_GPU_RESULT_v3.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(OUTPUT))
print('OFFICIAL PSD STRUCTURE PASS:',len(qa['psds']),'files')
#@markdown 기본 전달은 Cubism PRO 모델 템플릿이 적용되도록 준비된 PSD ZIP입니다.
DELIVERY_TARGET = 'cubism_pro' #@param ['cubism_pro', 'original_psd']
from google.colab import files
if DELIVERY_TARGET == 'cubism_pro':
    from tools.vts_cubism_pro import build_cubism_pro_handoff
    pro_packages = []
    for i, source_psd in enumerate(psds, 1):
        input_pack = ROOT / f'cubism_pro_input_{i:02d}.zip'
        with zipfile.ZipFile(input_pack, 'w', compression=zipfile.ZIP_DEFLATED) as payload:
            payload.write(source_psd, 'avatar.psd')
            for path, label in (
                (OUTPUT / 'PSD_STRUCTURE_QA.json', 'qa/PSD_STRUCTURE_QA.json'),
                (OUTPUT / 'run_meta.json', 'qa/run_meta.json'),
                (OUTPUT / 'full_character_input.png', 'references/full_character_input.png'),
            ):
                if path.is_file():
                    payload.write(path, label)
        pro_out = ROOT / f'Cubism_PRO_face_{i:02d}_Handoff.zip'
        outcome = build_cubism_pro_handoff(
            input_pack, pro_out, frame='upper', asset_kind='full_character')
        pro_packages.append(pro_out)
        print('PRO PSD handoff:', outcome['package'], 'native rigging: NOT DONE')
    if len(pro_packages) == 1:
        files.download(str(pro_packages[0]))
    else:
        bundle = ROOT / 'Cubism_PRO_all_PSD_candidates.zip'
        with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as out:
            for p in pro_packages:
                out.write(p, p.name)
        files.download(str(bundle))
else:
    files.download(str(archive))